# N-BEATS Model Inference

This notebook downloads the registered N-BEATS raw-input pipeline from W&B Model Registry, predicts directly on unprocessed `test.csv`, creates a Kaggle-ready submission, and logs inference artifacts to W&B.

In [ ]:
%pip install -q "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "torch>=2.3,<3" "cloudpickle>=3,<4" "matplotlib>=3.8,<4"

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Google Drive mount skipped. This is expected outside Colab.')

In [ ]:
import json
import os
from pathlib import Path

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb

WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
REGISTRY_ARTIFACT_URI = "wandb-registry-model/Walmart_NBEATS_Pipeline:latest"
PIPELINE_FILENAME = "walmart_nbeats_raw_pipeline.pkl"

DATA_DIR_CANDIDATES = [
    Path('/content/drive/MyDrive/walmart_competition_data'),
    Path('/content/drive/My Drive/walmart_competition_data'),
    Path('/content/walmart_competition_data'),
    Path('../../data'),
    Path('data'),
]
OUTPUT_DIR = Path('/content/artifacts/nbeats_inference') if Path('/content').exists() else Path('artifacts/nbeats_inference')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBMIT_TO_KAGGLE = False
KAGGLE_COMPETITION = "walmart-recruiting-store-sales-forecasting"
KAGGLE_MESSAGE = "W&B registered N-BEATS raw-input pipeline"

In [ ]:
def resolve_data_dir(candidates):
    for candidate in candidates:
        if candidate.exists() and (candidate / 'test.csv').exists():
            return candidate
    searched = '\n'.join(str(path) for path in candidates)
    raise FileNotFoundError(f'Could not find test.csv. Searched:\n{searched}')


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
test_raw = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['Date'])
print(f'Data directory: {DATA_DIR}')
print('test shape:', test_raw.shape)
print('test date range:', test_raw['Date'].min().date(), 'to', test_raw['Date'].max().date())

## Download registered pipeline

In [ ]:
wandb.login(key=os.environ.get('WANDB_API_KEY'), relogin=False)

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    job_type='nbeats_inference',
    name='NBEATS_Registry_Inference_Submission',
    config={
        'registry_artifact': REGISTRY_ARTIFACT_URI,
        'pipeline_filename': PIPELINE_FILENAME,
        'data_dir': str(DATA_DIR),
        'submit_to_kaggle': SUBMIT_TO_KAGGLE,
    },
)

model_artifact = run.use_artifact(REGISTRY_ARTIFACT_URI)
artifact_dir = Path(model_artifact.download(root=str(OUTPUT_DIR / 'registry_model')))
pipeline_path = artifact_dir / PIPELINE_FILENAME
if not pipeline_path.exists():
    raise FileNotFoundError(f'Pipeline file not found in artifact: {pipeline_path}')

with pipeline_path.open('rb') as file:
    pipeline = cloudpickle.load(file)

print(f'Loaded pipeline from: {pipeline_path}')

## Predict raw test and create submission

In [ ]:
predictions = pipeline.predict(test_raw)
if len(predictions) != len(test_raw):
    raise ValueError(f'Prediction row mismatch: {len(predictions)} predictions for {len(test_raw)} test rows')
if not np.isfinite(predictions).all():
    raise ValueError('Predictions contain non-finite values')

submission = test_raw[['Store', 'Dept', 'Date']].copy()
submission['Weekly_Sales'] = np.clip(predictions, a_min=0.0, a_max=None)
submission.insert(
    0,
    'Id',
    submission['Store'].astype(str) + '_' + submission['Dept'].astype(str) + '_' + submission['Date'].dt.strftime('%Y-%m-%d'),
)
submission = submission[['Id', 'Weekly_Sales']]

submission_path = OUTPUT_DIR / 'submission_nbeats_registry.csv'
submission.to_csv(submission_path, index=False)

print(f'Submission saved to: {submission_path}')
print('submission rows:', len(submission))
display(submission.head())

## Log inference diagnostics and submission artifact

In [ ]:
manifest = {
    'registry_artifact': REGISTRY_ARTIFACT_URI,
    'submission_path': str(submission_path),
    'submission_rows': int(len(submission)),
    'prediction_min': float(submission['Weekly_Sales'].min()),
    'prediction_mean': float(submission['Weekly_Sales'].mean()),
    'prediction_max': float(submission['Weekly_Sales'].max()),
    'prediction_std': float(submission['Weekly_Sales'].std()),
}
manifest_path = OUTPUT_DIR / 'nbeats_inference_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(submission['Weekly_Sales'], bins=80)
ax.set_title('N-BEATS registry submission prediction distribution')
ax.set_xlabel('Weekly_Sales')
ax.set_ylabel('Count')
plt.tight_layout()
hist_path = OUTPUT_DIR / 'nbeats_prediction_histogram.png'
fig.savefig(hist_path, dpi=160)
plt.show()

submission_artifact = wandb.Artifact('nbeats-kaggle-submission', type='submission')
submission_artifact.add_file(str(submission_path))
submission_artifact.add_file(str(manifest_path))
submission_artifact.add_file(str(hist_path))
run.log_artifact(submission_artifact, aliases=['latest'])

run.log({
    'inference/submission_rows': manifest['submission_rows'],
    'inference/prediction_min': manifest['prediction_min'],
    'inference/prediction_mean': manifest['prediction_mean'],
    'inference/prediction_max': manifest['prediction_max'],
    'inference/prediction_std': manifest['prediction_std'],
    'inference/submission_preview': wandb.Table(dataframe=submission.head(2000)),
    'inference/prediction_histogram': wandb.Image(str(hist_path)),
})
for key, value in manifest.items():
    run.summary[key] = value
run.finish()

print('Inference complete: registry model -> raw test -> submission artifact.')

## Optional Kaggle upload

In [ ]:
if SUBMIT_TO_KAGGLE:
    import subprocess
    subprocess.run(
        [
            'kaggle', 'competitions', 'submit',
            '-c', KAGGLE_COMPETITION,
            '-f', str(submission_path),
            '-m', KAGGLE_MESSAGE,
        ],
        check=True,
    )
    print('Kaggle submission uploaded.')
else:
    print('Kaggle upload skipped. The submission CSV is ready for manual upload.')